<a href="https://colab.research.google.com/github/cocyce459-oss/Equalizer314/blob/symeathrea/aistudio2api_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIStudio2API on Google Colab

**What this does:** Runs **Mag1cFall's** proxy (converts your Google AI Studio web session into a local OpenAI/Anthropic/Gemini-compatible API) on a free Colab VM, exposed via ngrok.

| Cell | What happens | Time |
|---|---|---|
| 1 | **Setup & Setting:** Mounts Drive, configures key, ngrok Authtoken | ~1min |
| 2 | Installation and Downloads One-time Google login via camoufox, noVNC screen (skips automatically if a saved login is found on Drive) | 2-5min |
| 3 | Starts API server | ~10s |
| 4 | Opens ngrok tunnel, prints public URL | ~10s |
| 5 | Tests the endpoint | ~5s |

**Important:**
- Colab free tier disconnects after ~90 minutes idle. Playing the infinite 1-second audio loop in cell below helps prevent it. Re-run cells 1, 3, and 4 after a reconnect.
- Your ngrok authtoken is embedded directly in **Cell 1** — do not share this notebook with anyone.
- Your AI Studio cookies and authentication states are securely stored on **YOUR** own Google Drive.
- **Exit status 255?** This happens if Camoufox fails to launch due to missing system libraries. **Cell 2 (Login)** automatically updates and installs all necessary display and browser packages (`libgtk-3-0`, `libasound2`, `libnss3`, etc.) before launching the browser.
- **Static URL:** Note that the ngrok public URL is keep because how it come along even whenever you restart the tunnel. You doesn't have to update your client's base URL. Just paste it with the endpoint printed by **Cell 4**.

In [22]:
#@title 1. Setup — download binary, mount Drive, configure tunnel {display-mode:"form"}
PROXY_API_KEY = "sk_aistudio_local" #@param {type:"string"}
ADMIN_PASSWORD = "admin123" #@param {type:"string"}
WAA_BACKEND = "spidermonkey" #@param ["spidermonkey", "camoufox"]
LISTEN_PORT = 2048 #@param {type:"integer"}
TUNNEL_PROVIDER = "cloudflared" #@param ["cloudflared", "ngrok"]
#@markdown If you select `ngrok`, provide your `NGROK_AUTHTOKEN`. Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "" #@param {type:"string"}

import os, subprocess, glob, shutil, sys, time, urllib.request, json as _json

WORK = "/content/aistudio2api"
AUTH_DIR = "/content/drive/MyDrive/aistudio2api/auth"

# --- Mount Drive ---
if not os.path.exists('/content/drive/MyDrive'):
    print("⌓ Mounting Google Drive (popup will appear)...")
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f"☑ Drive mount call returned an error: {e}. Checking if accessible anyway...")

if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError("Google Drive did not mount — accept the permission popup and re-run this cell.")
print("✓ Drive mounted")

os.makedirs(WORK, exist_ok=True)
os.makedirs(AUTH_DIR, exist_ok=True)

# --- Helpers ---
def run(cmd, **kw):
    """subprocess.run that surfaces stderr instead of swallowing failures."""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str, cmd))}\n{(r.stderr or '')[-2000:]}")
    return r


def download_extract(url, dest_path, workdir, label, force=False):
    """Download a .tar.gz and place its executable at dest_path."""
    if not force and os.path.isfile(dest_path) and os.access(dest_path, os.X_OK):
        print(f"✓ {label} already present: {dest_path}")
        return
    tgz = dest_path + ".download"
    print(f"⌓ Downloading {label} from {url}...")
    run(["curl", "-fSL", "--retry", "3", "-o", tgz, url])
    staging = os.path.join(workdir, ".staging_" + label)
    shutil.rmtree(staging, ignore_errors=True)
    os.makedirs(staging, exist_ok=True)
    try:
        run(["tar", "-xzf", tgz, "-C", staging])
    finally:
        if os.path.exists(tgz):
            os.remove(tgz)
    name = os.path.basename(dest_path)
    candidates = [p for p in glob.glob(os.path.join(staging, "**", name), recursive=True)
                  if os.path.isfile(p)]
    if not candidates:
        listing = glob.glob(os.path.join(staging, "**", "*"), recursive=True)[:20]
        raise FileNotFoundError(f"'{name}' not found inside the {label} archive. Archive contains: {listing}")
    if os.path.exists(dest_path):
        os.remove(dest_path)
    shutil.move(candidates[0], dest_path)
    shutil.rmtree(staging, ignore_errors=True)
    os.chmod(dest_path, 0o755)
    print(f"✓ {label} ready → {dest_path}")


# --- Dynamically fetch latest aistudio2api release version ---
def get_latest_aistudio2api_url():
    api_url = "https://api.github.com/repos/Mag1cFall/AIStudio2API/releases/latest"
    try:
        req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            data = _json.loads(response.read().decode())
            tag_name = data.get("tag_name", "v0.2.10")
            download_url = f"https://github.com/Mag1cFall/AIStudio2API/releases/download/{tag_name}/aistudio2api-{tag_name}-linux-amd64.tar.gz"
            print(f"✓ Detected latest version: {tag_name}")
            return download_url
    except Exception as e:
        print(f"☑ Failed to fetch latest version via GitHub API ({e}). Falling back to v0.2.10...")
        return "https://github.com/Mag1cFall/AIStudio2API/releases/download/v0.2.10/aistudio2api-v0.2.10-linux-amd64.tar.gz"

BIN_URL = get_latest_aistudio2api_url()
download_extract(BIN_URL, f"{WORK}/aistudio2api", WORK, "aistudio2api", force=True)

# --- Download selected tunnel client ---
if TUNNEL_PROVIDER == "cloudflared":
    dest_cf = f"{WORK}/cloudflared"
    if os.path.exists(dest_cf) and os.access(dest_cf, os.X_OK):
        print("✓ cloudflared binary already exists, skipping download.")
    elif shutil.which("cloudflared"):
        shutil.copy(shutil.which("cloudflared"), dest_cf)
        os.chmod(dest_cf, 0o755)
        print("✓ cloudflared copied from system")
    else:
        print("⌓ Downloading cloudflared...")
        run(["curl", "-L", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", dest_cf])
        os.chmod(dest_cf, 0o755)
        print("✓ cloudflared ready")
elif TUNNEL_PROVIDER == "ngrok":
    dest_ngrok = f"{WORK}/ngrok"
    if not NGROK_AUTHTOKEN.strip():
        print("☐— Warning: NGROK_AUTHTOKEN is empty. Ngrok will fail to launch without a valid token.")
    if os.path.exists(dest_ngrok) and os.access(dest_ngrok, os.X_OK):
        print("✓ ngrok binary already exists, skipping download.")
    else:
        print("⌓ Downloading ngrok...")
        # Fixed valid Ngrok v3 URLs with mirror fallback
        ngrok_urls = [
            "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz",
            "https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz"
        ]
        downloaded = False
        for url in ngrok_urls:
            try:
                run(["curl", "-fSL", "--retry", "2", "-o", f"{WORK}/ngrok.tgz", url])
                downloaded = True
                break
            except Exception:
                continue
        if not downloaded:
            raise RuntimeError("Failed to download Ngrok binary from all available mirrors.")

        run(["tar", "-xzf", f"{WORK}/ngrok.tgz", "-C", WORK])
        if os.path.exists(f"{WORK}/ngrok.tgz"):
            os.remove(f"{WORK}/ngrok.tgz")
        os.chmod(dest_ngrok, 0o755)
        print("✓ ngrok ready")
    # Configure token
    if NGROK_AUTHTOKEN.strip():
        run([dest_ngrok, "config", "add-authtoken", NGROK_AUTHTOKEN.strip()])
        print("✓ Ngrok authtoken configured")

# --- Write .env ---
env_content = f"""AISTUDIO_AUTH_STATES={AUTH_DIR}
LISTEN_ADDR=127.0.0.1:{LISTEN_PORT}
PROXY_API_KEY={PROXY_API_KEY}
ADMIN_AUTH_ENABLED=true
ADMIN_USERNAME=admin
ADMIN_PASSWORD={ADMIN_PASSWORD}
WAA_BACKEND={WAA_BACKEND}
AUTO_START=true
REQUEST_BODY_LOG=false
"""
with open(f"{WORK}/.env", "w") as f:
    f.write(env_content)
print("✓ .env written")

print(f"\nအ Binary: {WORK}/aistudio2api")
print(f"အ Auth state: {AUTH_DIR}")
print(f"အ .env: {WORK}/.env")

✓ Drive mounted
✓ Detected latest version: v0.3.1
⌓ Downloading aistudio2api from https://github.com/Mag1cFall/AIStudio2API/releases/download/v0.3.1/aistudio2api-v0.3.1-linux-amd64.tar.gz...
✓ aistudio2api ready → /content/aistudio2api/aistudio2api
✓ cloudflared binary already exists, skipping download.
✓ .env written

အ Binary: /content/aistudio2api/aistudio2api
အ Auth state: /content/drive/MyDrive/aistudio2api/auth
အ .env: /content/aistudio2api/.env


In [ ]:
#@title 2. One-time Google login via noVNC {display-mode:"form"}
#@markdown Opens a browser screen you can access from any device via the printed URL. Sign into your Google account, wait for AI Studio to load — the cell **detects the saved login itself and cleans up** (no need to interrupt it).
#@markdown If a login is already saved on Drive this cell exits immediately, unless FORCE_RELOGIN is set.
FORCE_RELOGIN = False #@param {type:"boolean"}
LOGIN_TIMEOUT_MIN = 15 #@param {type:"integer"}

import os, subprocess, time, signal, threading, shutil, glob, urllib.request, re, json as _json

WORK = "/content/aistudio2api"
AUTH_DIR = "/content/drive/MyDrive/aistudio2api/auth"
VNC_DISPLAY = ":99"
VNC_PORT = 5900
WEBSOCK_PORT = 6080
NGROK_API_PORT = 4040
NOVNC_DIR = "/usr/share/novnc"
PROVIDER = globals().get("TUNNEL_PROVIDER", "cloudflared")


def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str, cmd))}\n{(r.stderr or '')[-2000:]}")
    return r


def login_snapshot():
    """Saved accounts = auth/<Google email>/storage-state.json → {path: mtime}."""
    return {p: os.path.getmtime(p)
            for p in glob.glob(os.path.join(AUTH_DIR, "*", "storage-state.json"))}


# --- Pre-check #1: is a Google login already saved on Drive? ---
before = login_snapshot()
if before and not FORCE_RELOGIN:
    print("✓ Google login already saved — skipping browser login.")
    for p in before:
        print("   -", p.split("/")[-2])
    print("  (set FORCE_RELOGIN=True to add or re-login an account)")
else:
    # --- Install Display Stack + Camoufox Runtime Packages (Only in Cell 2) ---
    DISPLAY_PKGS = ["xvfb", "x11vnc", "fluxbox", "websockify", "novnc", "wget"]
    CAMOUFOX_LIBS = ["libgtk-3-0", "libasound2", "libnss3", "libdbus-glib-1-2",
                     "libxtst6", "libxrandr2", "libgbm1", "libxkbcommon0",
                     "libpango-1.0-0", "libcairo2", "libxcomposite1", "libxdamage1",
                     "libxfixes3", "fonts-liberation"]
    ALIASES = {"libasound2": ["libasound2t64"], "libgtk-3-0": ["libgtk-3-0t64"]}

    print("⌖ Updating system package repository lists to retrieve latest available modules...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True, text=True)

    print("⌖ Installing packages (display stack + Firefox runtime for Camoufox)...")
    r = subprocess.run(["apt-get", "install", "-y", "--only-upgrade", "-qq"] + DISPLAY_PKGS + CAMOUFOX_LIBS,
                       capture_output=True, text=True)

    r = subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", "-qq"] + DISPLAY_PKGS + CAMOUFOX_LIBS,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("Installing package by package...")
        unresolved = []
        for pkg in DISPLAY_PKGS + CAMOUFOX_LIBS:
            done = False
            for cand in [pkg] + ALIASES.get(pkg, []):
                rr = subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", "-qq", cand],
                                    capture_output=True, text=True)
                if rr.returncode == 0:
                    done = True
                    break
            if not done:
                unresolved.append(pkg)
        if unresolved:
            raise RuntimeError("Could not install required packages: " + ", ".join(unresolved)
                               + "\nCamoufox login WILL fail with exit 255 without them.")
    print("✓ System packages installed and updated to their latest available versions")

    procs = []
    try:
        # --- Kill any previous session ---
        for proc_name in ["aistudio2api", "Xvfb", "x11vnc", "websockify", "fluxbox", "ngrok", "cloudflared"]:
            subprocess.run(["pkill", "-f", proc_name], capture_output=True)
        time.sleep(2)

        # --- Xvfb + fluxbox ---
        print("⌖ Starting virtual display...")
        procs.append(subprocess.Popen(
            ["Xvfb", VNC_DISPLAY, "-screen", "0", "1280x900x24", "-ac", "+extension", "GLX", "+render", "-noreset"],
            env={**os.environ, "DISPLAY": VNC_DISPLAY},
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
        time.sleep(2)
        procs.append(subprocess.Popen(["fluxbox"], env={**os.environ, "DISPLAY": VNC_DISPLAY},
                                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
        time.sleep(1)
        print("✓ Virtual display + window manager running")

        # --- x11vnc ---
        procs.append(subprocess.Popen(
            ["x11vnc", "-display", VNC_DISPLAY, "-forever", "-nopw", "-quiet",
             "-rfbport", str(VNC_PORT), "-shared", "-noxdamage", "-noxrecord", "-noxfixes"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
        time.sleep(2)

        # --- noVNC static files BEFORE websockify serves them ---
        if not os.path.isfile(os.path.join(NOVNC_DIR, "vnc.html")):
            print("⌖ Downloading noVNC...")
            run(["wget", "-q", "https://github.com/novnc/noVNC/archive/refs/tags/v1.4.0.tar.gz",
                 "-o", "/tmp/novnc.tar.gz"])
            shutil.rmtree("/tmp/novnc", ignore_errors=True)
            os.makedirs("/tmp/novnc", exist_ok=True)
            run(["tar", "-xzf", "/tmp/novnc.tar.gz", "-C", "/tmp/novnc", "--strip-components=1"])
            shutil.rmtree(NOVNC_DIR, ignore_errors=True)
            shutil.copytree("/tmp/novnc", NOVNC_DIR)
            print("✓ noVNC installed")

        # --- websockify (after the noVNC files exist) ---
        procs.append(subprocess.Popen(
            ["websockify", "--web", NOVNC_DIR, f"0.0.0.0:{WEBSOCK_PORT}", f"localhost:{VNC_PORT}"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
        time.sleep(1)
        print(f"✓ noVNC server on port {WEBSOCK_PORT}")

        # --- Tunnel for the browser screen ---
        vnc_url = None
        if PROVIDER == "cloudflared":
            cf_log = open(f"{WORK}/cloudflared_vnc.log", "w")
            procs.append(subprocess.Popen(
                [f"{WORK}/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{WEBSOCK_PORT}"],
                stdout=cf_log, stderr=subprocess.STDOUT))
            time.sleep(5)
            for _ in range(15):
                if os.path.exists(f"{WORK}/cloudflared_vnc.log"):
                    with open(f"{WORK}/cloudflared_vnc.log", "r") as f:
                        log_content = f.read()
                    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", log_content)
                    if match:
                        vnc_url = match.group(0)
                        break
                time.sleep(1)
        else:
            ngrok_log = open(f"{WORK}/ngrok_vnc.log", "w")
            procs.append(subprocess.Popen(
                [f"{WORK}/ngrok", "http", str(WEBSOCK_PORT), "--log=stdout", "--log-format=json"],
                stdout=ngrok_log, stderr=subprocess.STDOUT))
            time.sleep(3)
            for _ in range(15):
                try:
                    with urllib.request.urlopen(f"http://127.0.0.1:{NGROK_API_PORT}/api/tunnels", timeout=2) as r:
                        data = _json.loads(r.read())
                    for t in data.get("tunnels", []):
                        if t.get("proto") == "https":
                            vnc_url = t["public_url"]
                            break
                except Exception:
                    pass
                time.sleep(1)

        if vnc_url:
            print(f"\n{'='*60}")
            print("║─ OPEN THIS URL IN YOUR BROWSER (works from any device):")
            print(f"   {vnc_url}/vnc.html?autoconnect=true&resize=scale")
            print(f"{'='*60}\n")
            print("1. Open the URL (cloudflared or ngrok may show a warning page — click 'Visit Site' or 'Continue')")
            print("2. Click Connect in the noVNC page")
            print("3. Sign into your Google account in the browser window that appears")
            print("4. Wait until AI Studio is fully loaded — this cell detects the saved login and stops itself")
        else:
            print(f"☐ Could not get {PROVIDER} URL — check configuration / logs in {PROVIDER}_vnc.log")

        # --- Run setup --login ---
        print("\n⌖ Launching aistudio2api setup --login...")
        setup_proc = subprocess.Popen(
            [f"{WORK}/aistudio2api", "setup", "--login"],
            cwd=WORK,
            env={**os.environ, "DISPLAY": VNC_DISPLAY,
                 "AISTUDIO_AUTH_STATES": AUTH_DIR, "WAA_BACKEND": "camoufox"},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        log_lines = []

        def pump():
            for line in setup_proc.stdout:
                log_lines.append(line)
                print(line, end="")

        threading.Thread(target=pump, daemon=True).start()

        deadline = time.time() + LOGIN_TIMEOUT_MIN * 60
        try:
            while time.time() < deadline:
                if login_snapshot() != before:
                    print("\n✓ Account saved to Drive!")
                    time.sleep(5)
                    break
                if setup_proc.poll() is not None:
                    break
                time.sleep(2)
        except KeyboardInterrupt:
            print("\n☐ Interrupted — verifying whether the login was saved...")

        if setup_proc.poll() is None:
            setup_proc.send_signal(signal.SIGINT)
            try:
                setup_proc.wait(timeout=20)
            except subprocess.TimeoutExpired:
                setup_proc.kill()
        rc = setup_proc.returncode

        after = login_snapshot()
        if after != before:
            print(f"\n{'='*60}")
            print(f"✅ LOGIN SAVED — accounts in {AUTH_DIR}:")
            for p in after:
                print("   -", p.split("/")[-2])
            print(f"{'='*60}")
            print("→ Run Cell 3 next. This cell is only needed once (login persists on Drive).")
        else:
            print(f"\n✗ Login NOT saved (setup exit code: {rc})")
            print("--- last setup output ---")
            print("".join(log_lines[-30:]) or "(no output)")
            print("-------------------------")
    finally:
        # --- Always tear down the VNC stack ---
        for p in procs:
            try:
                p.send_signal(signal.SIGTERM)
            except Exception:
                pass
        for name in ["Xvfb", "x11vnc", "websockify", "ngrok", "cloudflared"]:
            subprocess.run(["pkill", "-f", name], capture_output=True)
        print("✓ VNC session cleaned up")

In [34]:
#@title 3. Start the API server {display-mode:"form"}
import os, subprocess, time, urllib.request

WORK = "/content/aistudio2api"
AUTH_DIR = "/content/drive/MyDrive/aistudio2api/auth"
PORT = globals().get("LISTEN_PORT")
LOG = f"{WORK}/server.log"

if not os.path.isfile(f"{WORK}/aistudio2api"):
    raise RuntimeError("Binary not found — run Cell 1 first.")
accounts = [p for p in __import__("glob").glob(os.path.join(AUTH_DIR, "*", "storage-state.json"))]
if not accounts:
    print("⚠️ No logged-in account found on Drive — run Cell 2 first (the API won't serve without one).")

# Kill previous server
subprocess.run(["pkill", "-f", "aistudio2api.*--listen"], capture_output=True)
time.sleep(2)

# Launch server (log to file so a failure can be reported without blocking)
logf = open(LOG, "w")
server_proc = subprocess.Popen(
    [f"{WORK}/aistudio2api", "--listen", f"127.0.0.1:{PORT}", "--auth", AUTH_DIR],
    cwd=WORK,
    env={**os.environ},
    stdout=logf, stderr=subprocess.STDOUT)

# Wait for it to be ready
ready = False
for _ in range(30):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)

logf.flush()
tail = open(LOG).read()[-3000:]
if ready:
    print(f"✓ API server running on http://127.0.0.1:{PORT}")
    print(f"  - Health:  http://127.0.0.1:{PORT}/health")
    print(f"  - Admin UI: http://127.0.0.1:{PORT} (user=admin, pass={globals().get('ADMIN_PASSWORD', 'admin123')})")
    print(f"  - Auth dir: {AUTH_DIR}")
else:
    if server_proc.poll() is not None:
        print(f"✗ Server exited with code {server_proc.returncode}")
    else:
        print("⚠️ Server not ready yet — it is still starting.")
    print("--- server log tail ---")
    print(tail or "(log empty)")

✓ API server running on http://127.0.0.1:2048
  - Health:  http://127.0.0.1:2048/health
  - Admin UI: http://127.0.0.1:2048 (user=admin, pass=admin123)
  - Auth dir: /content/drive/MyDrive/aistudio2api/auth


In [41]:
#@title 4. Open selected tunnel & get public URL {display-mode:"form"}
import os, subprocess, time, re, urllib.request, socket, json as _json

WORK = "/content/aistudio2api"
PORT = globals().get("LISTEN_PORT")
PROVIDER = globals().get("TUNNEL_PROVIDER", "cloudflared")
BIN_PATH = f"{WORK}/aistudio2api"

def is_port_listening(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

# Auto-start backend if it isn't running
if not is_port_listening(PORT):
    print(f"⚠️ aistudio2api is not running on 127.0.0.1:{PORT}. Auto-starting it now...")
    if os.path.exists(BIN_PATH):
        os.chmod(BIN_PATH, 0o755)
        log_file = open(f"{WORK}/service.log", "a")
        subprocess.Popen([BIN_PATH, "-open-ui=false"], cwd=WORK, stdout=log_file, stderr=subprocess.STDOUT)
        for _ in range(12):
            if is_port_listening(PORT):
                print(f"✓ aistudio2api started on 127.0.0.1:{PORT}")
                break
            time.sleep(1)

if not is_port_listening(PORT):
    print("❌ Error: Local service could not be reached. Checking service.log:")
    if os.path.exists(f"{WORK}/service.log"):
        print(open(f"{WORK}/service.log").read()[-2000:])
    raise RuntimeError("Backend service is down. Please verify Cell 2.")

# Restart tunnel clean
subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok http"], capture_output=True)
time.sleep(2)

url = None
tunnel_file = f"{WORK}/tunnel_url.txt"
if os.path.exists(tunnel_file):
    os.remove(tunnel_file)

if PROVIDER == "cloudflared":
    CLOUDFLARED_BIN = f"{WORK}/cloudflared"
    if not os.path.exists(CLOUDFLARED_BIN):
        raise RuntimeError("cloudflared binary not found! Please run Cell 1 setup first.")
    os.chmod(CLOUDFLARED_BIN, 0o755)

    cf_log_path = f"{WORK}/cloudflared.log"
    cf_log = open(cf_log_path, "w")
    cf_proc = subprocess.Popen(
        [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=cf_log, stderr=subprocess.STDOUT
    )

    print("⌓ Connecting to Cloudflare edge...")
    registered = False
    for _ in range(30):
        if cf_proc.poll() is not None:
            break
        if os.path.exists(cf_log_path):
            with open(cf_log_path, "r") as f:
                log_content = f.read()
            matches = re.findall(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", log_content)
            if matches:
                url = matches[-1]
            if "Registered tunnel connection" in log_content or "connection registered" in log_content:
                registered = True
            if url and registered:
                break
        time.sleep(1)

    if url:
        print("⏳ Waiting 4s for Cloudflare DNS edge propagation...")
        time.sleep(4)

elif PROVIDER == "ngrok":
    NGROK_BIN = f"{WORK}/ngrok"
    if not os.path.exists(NGROK_BIN):
        raise RuntimeError("ngrok binary not found! Please run setup first.")

    ngrok_log_path = f"{WORK}/ngrok.log"
    ngrok_log = open(ngrok_log_path, "w")
    ng_proc = subprocess.Popen(
        [NGROK_BIN, "http", str(PORT), "--log=stdout", "--log-format=json"],
        stdout=ngrok_log, stderr=subprocess.STDOUT
    )

    print("⌓ Connecting to Ngrok...")
    for _ in range(25):
        if ng_proc.poll() is not None:
            break
        try:
            req = urllib.request.Request("http://127.0.0.1:4040/api/tunnels", headers={"User-Agent": "curl/7.68.0"})
            with urllib.request.urlopen(req, timeout=2) as r:
                data = _json.loads(r.read())
            for t in data.get("tunnels", []):
                if t.get("proto") == "https":
                    url = t["public_url"]
                    break
            if url:
                break
        except Exception:
            pass
        time.sleep(1)

if url:
    with open(tunnel_file, "w") as f:
        f.write(url)
    globals()["PUBLIC_TUNNEL_URL"] = url
    api_key = globals().get("PROXY_API_KEY", "sk_aistudio_local")
    print(f"\n{'='*65}")
    print(f"🎉 PUBLIC TUNNEL READY (via {PROVIDER.upper()}):")
    print(f"{'='*65}")
    print(f"  Tunnel URL /")
    print(f"  Admin UI:      {url} (login: admin / your ADMIN_PASSWORD)")
    print(f"  API base (v1): {url}/v1")
    print(f"  API Key:       {api_key}")
    print(f"{'='*65}\n")
else:
    print(f"❌ Could not establish {PROVIDER} tunnel. Logs:")
    log_file = f"{WORK}/{PROVIDER}.log"
    if os.path.exists(log_file):
        print(open(log_file).read()[-2000:])

⚠️ aistudio2api is not running on 127.0.0.1:2048. Auto-starting it now...
✓ aistudio2api started on 127.0.0.1:2048
⌓ Connecting to Cloudflare edge...
⏳ Waiting 4s for Cloudflare DNS edge propagation...

🎉 PUBLIC TUNNEL READY (via CLOUDFLARED):
  Tunnel URL /
  Admin UI:      https://ata-origins-candy-county.trycloudflare.com (login: admin / your ADMIN_PASSWORD)
  API base (v1): https://ata-origins-candy-county.trycloudflare.com/v1
  API Key:       sk_aistudio_local



In [43]:
#@title 5. Test the endpoint {display-mode:"form"}
import urllib.request, urllib.error, json, os, re, time

WORK = "/content/aistudio2api"
PORT = globals().get("LISTEN_PORT", 2048)
PROVIDER = globals().get("TUNNEL_PROVIDER", "cloudflared")
API_KEY = globals().get("PROXY_API_KEY", "sk_aistudio_local")

# --- 1. Resolve Public Tunnel (Strict - Anti-Cheat) ---
BASE = globals().get("PUBLIC_TUNNEL_URL")

if not BASE and os.path.exists(f"{WORK}/tunnel_url.txt"):
    with open(f"{WORK}/tunnel_url.txt", "r") as f:
        BASE = f.read().strip()

if not BASE and PROVIDER == "ngrok":
    try:
        req_ng = urllib.request.Request("http://127.0.0.1:4040/api/tunnels", headers={"User-Agent": "curl/7.68.0"})
        with urllib.request.urlopen(req_ng, timeout=3) as r:
            data = json.loads(r.read())
        for t in data.get("tunnels", []):
            if t.get("proto") == "https":
                BASE = t["public_url"]
                break
    except Exception:
        pass

if not BASE and PROVIDER == "cloudflared" and os.path.exists(f"{WORK}/cloudflared.log"):
    with open(f"{WORK}/cloudflared.log", "r") as f:
        matches = re.findall(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", f.read())
        if matches:
            BASE = matches[-1] # Pick the newest active URL

# ANTI-CHEAT CHECK: Never silently fallback to localhost
if not BASE or "127.0.0.1" in BASE or "localhost" in BASE:
    raise RuntimeError(
        f"❌ NO VALID PUBLIC TUNNEL FOUND FOR '{PROVIDER.upper()}'!\n"
        f"Detected endpoint: {BASE or 'None'}\n"
        f"Anti-cheat protection: Refusing to fall back to localhost.\n"
        f"Please run Cell 4 to establish an active public tunnel first."
    )

print(f"🌐 Testing against PUBLIC TUNNEL: {BASE}\n")

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "ngrok-skip-browser-warning": "true"
}

# --- 2. Helper with Cloudflare Error 1033 Auto-Retry ---
def fetch_json(url, data=None, timeout=60, max_retries=6):
    req = urllib.request.Request(url, data=data, headers=HEADERS)
    for attempt in range(1, max_retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode())
        except urllib.error.HTTPError as e:
            body = ""
            try:
                body = e.read().decode()
            except Exception:
                pass

            # Handle Cloudflare Error 1033 (Edge propagation lag)
            if e.code == 530 and "1033" in body:
                if attempt < max_retries:
                    print(f"  ⏳ [Cloudflare Error 1033] Edge is still propagating routes... Retrying in 4s ({attempt}/{max_retries})")
                    time.sleep(4)
                    continue
            raise RuntimeError(f"HTTP {e.code} ({e.reason}):\n{body}")
        except Exception as e:
            if attempt < max_retries and ("reset" in str(e).lower() or "refused" in str(e).lower()):
                time.sleep(3)
                continue
            raise e

# --- 3. Test /v1/models ---
print(f"📡 1/2. Querying {BASE}/v1/models ...")
try:
    models_res = fetch_json(f"{BASE}/v1/models")
    models_list = models_res.get("data", [])
    if not models_list:
        raise ValueError(f"No models returned from backend! Output: {models_res}")

    print(f"✓ Success! Found {len(models_list)} live models on this server:")
    for m in models_list[:6]:
        print(f"   • {m['id']}")
    if len(models_list) > 6:
        print(f"   ... and {len(models_list) - 6} more")
except Exception as e:
    print(f"✗ Failed to list models: {e}")
    raise SystemExit(1)

# --- 4. Test Chat Completion with a Real, Live Model ---
# Automatically select an available model
available_ids = [m["id"] for m in models_list]
preferred = ["gemini-3.5-flash-lite", "gemini-flash-lite-latest"]
target_model = next((p for p in preferred if p in available_ids), available_ids[0])

print(f"\n💬 2/2. Testing chat completion through public tunnel with '{target_model}' ...")
payload = json.dumps({
    "model": target_model,
    "messages": [
        {"role": "user", "content": "Hello! Reply with exactly: 'Tunnel test successful!'"}
    ],
    "stream": False
}).encode("utf-8")

try:
    completion_res = fetch_json(f"{BASE}/v1/chat/completions", data=payload, timeout=60)
    reply = completion_res["choices"][0]["message"]["content"]
    print(f"✓ REAL MODEL RESPONSE (verified via {PROVIDER.upper()} tunnel):")
    print(f"-------------------------------------------------------------")
    print(f"{reply.strip()}")
    print(f"-------------------------------------------------------------")
    print(f"🎉 Tunnel is fully operational and authenticated!")
except Exception as e:
    print(f"✗ Completion failed: {e}")

🌐 Testing against PUBLIC TUNNEL: https://ata-origins-candy-county.trycloudflare.com

📡 1/2. Querying https://ata-origins-candy-county.trycloudflare.com/v1/models ...
✓ Success! Found 73 live models on this server:
   • gemini-3.8-flash
   • gemini-3.8-flash-search
   • gemini-3.8-flash-tts
   • gemini-3.5-flash-lite
   • gemini-3.5-flash-lite-search
   • gemini-3.1-pro-preview
   ... and 67 more

💬 2/2. Testing chat completion through public tunnel with 'gemini-3.5-flash-lite' ...
✓ REAL MODEL RESPONSE (verified via CLOUDFLARED tunnel):
-------------------------------------------------------------
Tunnel test successful!
-------------------------------------------------------------
🎉 Tunnel is fully operational and authenticated!


In [ ]:
#@title Keep Alive — Infinite Silent Audio {display-mode: "form"}
#@markdown Toggle to run or deactivate a silent audio loop in the background to prevent Colab from disconnecting.
Active = True #@param {type:"boolean"}

import time
from IPython.display import display, Audio, HTML
import numpy as np

if Active:
    print("🚀 Keep-alive script started. Running in background...")
    # Generate 1 second of silence (frequency 0)
    sr = 22050
    silent_wave = np.zeros(sr)

    try:
        while True:
            # Displays an invisible, silent audio player that auto-plays
            display(Audio(silent_wave, rate=sr, autoplay=True))
            time.sleep(60)
    except KeyboardInterrupt:
        print("\n🛑 Keep-alive stopped.")
else:
    display(HTML('''
    <p style="color: gray; font-size: 11px; margin-top: 4px;">⏸️ Infinite silent audio is deactivated.</p>
    '''))

## 6️⃣ Using your endpoint

**OpenAI-compatible clients** (Cursor, Continue.dev, OpenWebUI, etc.):
```
Base URL: https://xxxx.ngrok-free.app/v1
API Key:  sk-aistudio-local
Model:    pick from /v1/models (e.g. gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-pro-preview)
```

**curl:**
```bash
curl https://xxxx.ngrok-free.app/v1/chat/completions \
  -H "Authorization: Bearer sk-aistudio-local" \
  -H "Content-Type: application/json" \
  -H "ngrok-skip-browser-warning: true" \
  -d '{"model":"gemini-3.8-flash","messages":[{"role":"user","content":"hi"}]}'
```

**Python:**
```python
from openai import OpenAI
client = OpenAI(base_url="https://xxxx.ngrok-free.app/v1", api_key="sk-aistudio-local")
r = client.chat.completions.create(model="gemini-3.8-flash",
    messages=[{"role":"user","content":"Hello!"}])
print(r.choices[0].message.content)
```

### After a Colab disconnect
1. Re-run **Cell 1** (re-downloads binary, remounts Drive, re-installs packages — about 1min)
2. Re-run **Cell 3** (starts server using saved login from Drive)
3. Re-run **Cell 4** (keeped ngrok URL — the URL stays every reconnect)
4. ~~Cell 2~~ not needed — login persists in `MyDrive/aistudio2api/auth/<email>/storage-state.json`, and Cell 2 skips itself when it finds one, except if `FORCE_RELOGIN` are active.

### Notes
- The ngrok URL **persist on every reconnect**. So your client's Base URL stays each time.
- Free ngrok shows an interstitial warning page for browser access; API calls with the `ngrok-skip-browser-warning` header or non-browser clients are unaffected.
- Admin UI login: `admin` / the ADMIN_PASSWORD you set in Cell 1. Otherwise, you enter `admin` in the user only (Password isn't set).
- Multi-account: set `FORCE_RELOGIN=True` in Cell 2 to add another account (one folder per account under `MyDrive/aistudio2api/auth/`, each with its own `storage-state.json`) to round-robin quotas.
- Cell 2 verifies `storage-state.json` actually appeared before claiming success.